# B · Anatomical topology and mask-duration controls

This experiment tests why a graph-time mask might help. Connected body regions change both which joints are missing and the duration of the missing interval, so the graph arm needs controls that separate those properties.

Read after notebooks 00–06. This notebook uses the prepared bundle and saved evaluation from that same run; the internal recipe group is `T`.


In [ ]:
from pathlib import Path
import json, os, sys

# Find the checkout/release from the notebook's working directory.
ROOT = Path(os.environ.get('GF_ROOT', Path.cwd())).resolve()
while not (ROOT / 'src/gavd6_sjepa').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'src/gavd6_sjepa').is_dir(), 'Open this notebook from the GAVD6 checkout or release.'
sys.path.insert(0, str(ROOT / 'notebooks/gait_fidelity'))
sys.path.insert(0, str(ROOT / 'src'))
from tutorial_helpers import configure, preview_images
study = configure(ROOT)


## Inspect the exact recipe cells

Compare the shuffled-topology and random-joint-interval recipes with the graph-time recipes in experiment A under the same encoder, output objective and seed. Shuffled topology changes the connections defining a region; random-joint intervals use the same nominal duration distribution while removing anatomical connectivity; realized exposure and run lengths still require auditing.


In [ ]:
import pandas as pd
plan = study.artifact('plan.json')
group_recipes = [r for r in plan['recipes'] if r['group'] == 'T']
expected_count = 8 if plan.get('experiment_set', 'full') == 'full' else {'M': 4, 'T': 0, 'P': 2, 'I': 4, 'L': 0}['T']
assert len(group_recipes) == expected_count, 'Saved plan differs from the selected experiment set.'
recipe_ids = {r['recipe_id'] for r in group_recipes}
display(pd.DataFrame(group_recipes))
print('Final models:', len(group_recipes) * len(plan['seeds']), 'Seeds:', plan['seeds'])
if not group_recipes:
    print('This group is outside the saved core protocol. Its worked examples are educational; no results are implied.')


## Measure the remaining differences between masking policies

For token $k$, estimate its hiding probability by
$\hat p_k=\sum_d H_{d,k}/\sum_d O_{d,k}$, where $O$ indicates a token with
at least one observed frame. Equal hidden-token counts do not imply equal
$\hat p_k$. Anatomical regions have different joint memberships, and overlap
or truncation changes the final lengths of hidden intervals.

We use a small deterministic training subset to make the calculation visible.
This tutorial audit is separate from the retained full-run audit. Linear run
lengths describe the crop seen by the encoder; a cyclic interval crossing
the crop boundary contributes two runs. The normalized Wasserstein distance
measures the average displacement between these run-length distributions,
divided by the number of temporal patches.


In [ ]:
import numpy as np
from scipy.stats import wasserstein_distance
from gavd6_sjepa.research_directions.gait_fidelity.data import load_dataset
from gavd6_sjepa.research_directions.gait_fidelity.masking import sample_mask, audit_matching
bundle = load_dataset(study.bundle_path()).subset('train')
count = min(6, len(bundle.records))
selected_rows = np.linspace(0, len(bundle.records) - 1, count, dtype=int)
observed = bundle.inputs['observed'][selected_rows]
cfg = study.artifact('config.json')
patch = cfg['model']['patch_size']
fraction = cfg['training']['mask_fraction']
available = observed.reshape(count, -1, patch, 12).any(axis=2)
draws, seed = 32, 17
probabilities, runs, budgets = {}, {}, {}
for policy in ['graph_time', 'shuffled_topology', 'random_joint_intervals']:
    rng = np.random.default_rng(seed)
    bank = np.stack([sample_mask(observed, policy, rng=rng, fraction=fraction,
                                 patch_size=patch) for _ in range(draws)])
    denominator = available.sum(axis=0) * draws
    probabilities[policy] = np.divide(bank.sum(axis=(0, 1)), denominator,
        out=np.zeros_like(denominator, dtype=float), where=denominator > 0)
    budgets[policy] = bank.sum(axis=(-1, -2))
    lengths = []
    for row in bank.reshape(-1, *available.shape[1:]):
        for joint in range(12):
            transitions = np.diff(np.r_[False, row[:, joint], False].astype(int))
            lengths.extend((np.flatnonzero(transitions == -1) - np.flatnonzero(transitions == 1)).tolist())
    runs[policy] = lengths
reference = audit_matching(observed, draws=draws, seed=seed, fraction=fraction, patch_size=patch)
rows = []
for policy in ['shuffled_topology', 'random_joint_intervals']:
    usable_slots = denominator > 0
    maximum = (float(abs(probabilities[policy] - probabilities['graph_time'])[usable_slots].max())
               if usable_slots.any() else None)
    distance = (wasserstein_distance(runs[policy], runs['graph_time']) / available.shape[1]
                if runs[policy] and runs['graph_time'] else None)
    for value, name in [(maximum, 'maximum_joint_time_probability_difference'),
                        (distance, 'normalized_run_length_Wasserstein')]:
        expected = reference['comparisons'][policy][name]
        if value is None:
            assert expected is None  # Unsupported is not a successful match.
        else:
            np.testing.assert_allclose(value, expected)
    assert np.array_equal(budgets[policy], budgets['graph_time'])
    rows.append({'control': policy, 'max_probability_difference': maximum,
                 'normalized_run_distance': distance,
                 'tutorial_default_tolerance_passed': reference['comparisons'][policy]['tolerance_passed']})
display(pd.DataFrame(rows))
print('These 32-draw examples reproduce the audit calculation; inspect the retained run audit too.')


A failed matching tolerance leaves joint exposure or persistence as an
alternative explanation for a performance difference. The production audit
retains that failure and the control's predictions. Do not drop a control
or adjust its tolerance after seeing which method wins.


## Follow the shared dependencies

This tutorial inspects the existing central queue. It does not launch a separate copy of its group: that would duplicate pretraining and break the global budget. Notebook 04 launches all groups, and this table identifies the phases that belong to the present comparison.


In [ ]:
final_phases = [p for p in plan['phases'] if p['phase'] != 'pretrain' and p['recipe']['recipe_id'] in recipe_ids]
parent_ids = {parent for p in final_phases for parent in p['depends_on'] if parent != 'prepare'}
selected = [p for p in plan['phases'] if p in final_phases or p['phase_id'] in parent_ids]
display(pd.DataFrame([{'phase_id': p['phase_id'], 'phase': p['phase'], 'seed': p['seed'],
                      'depends_on': ', '.join(p['depends_on'])} for p in selected]))


## Inspect completed checkpoints and learning histories

Each completed phase links its retained checkpoint, history and predictions to its source identity. Missing phases are reported as pending; a checkpoint from another study is not substituted.


In [ ]:
ledger_path = study.work / 'ledger.json'
completed = json.loads(ledger_path.read_text()).get('completed', {}) if ledger_path.exists() else {}
rows = []
for phase in selected:
    saved = completed.get(phase['phase_id'])
    result = saved.get('result', {}) if saved else {}
    rows.append({'phase_id': phase['phase_id'], 'status': 'complete' if saved else 'pending',
                 'checkpoint': result.get('checkpoint'), 'predictions': result.get('predictions')})
display(pd.DataFrame(rows))
history_candidates = []
for row in rows:
    if row['checkpoint']:
        history_path = Path(row['checkpoint']).parent / 'history.json'
        if history_path.exists(): history_candidates.append(history_path)
if history_candidates:
    history_path = history_candidates[0]
    print('First declared completed history:', history_path)
    display(pd.DataFrame(json.loads(history_path.read_text())))
else:
    print('No completed histories yet. Run or resume the central queue from notebook 04.')


## Read group results on the common population

Inspect actual per-joint hide frequencies, temporal run lengths and retained visible support. Equal average mask fraction alone does not establish a matched masking task. A topology effect that disappears after matching visibility should be described at that narrower level.


In [ ]:
person_path = study.work / 'evaluation/per-person.csv'
if person_path.exists():
    people = pd.read_csv(person_path)
    display(people.loc[people['method'].isin(recipe_ids)])
    coverage_path = study.work / 'evaluation/coverage.csv'
    if coverage_path.exists():
        coverage = pd.read_csv(coverage_path)
        display(coverage.loc[coverage['method'].isin(recipe_ids)])
else:
    print('Evaluation is pending. These recipe cards do not fabricate or extrapolate results.')


Read the matching controls from the other experiment tutorials before attribution. Full evaluation and numerical reconstruction are covered by notebooks 05 and 06.
